# Silver — Exchange Rates & OMX

Cleans Bronze market data: drops null rows, derives ISK/EUR cross-rate, and writes a production-ready table.

**Source:** `bronze.yahoo_finance_raw`  
**Output:** `silver.exchange_rates` (Delta table)  
**Key derivation:** `close_iskeur = close_iskusd_x / close_eurusd_x`  

In [ ]:
df = spark.read.format("delta").table("bronze.yahoo_finance_raw")

print(f"Bronze rows: {df.count()}")

In [ ]:
df_silver = spark.sql("""
    SELECT
        *,
        CASE
            WHEN close_eurusd_x != 0
            THEN ROUND(close_iskusd_x / close_eurusd_x, 6)
            ELSE NULL
        END                      AS close_iskeur,
        CURRENT_TIMESTAMP()      AS processed_at
    FROM bronze.yahoo_finance_raw
    WHERE close_eurusd_x  IS NOT NULL
      AND close_iskusd_x  IS NOT NULL
      AND close_eurusd_x  != 0
      AND close_iskusd_x  != 0
""")

if df_silver.isEmpty():
    raise ValueError("[silver_yahoo_finance] No rows after cleaning - halting.")

df_silver.createOrReplaceTempView("v_fx_silver")

In [ ]:
if not spark.catalog.tableExists("silver.exchange_rates"):
    spark.sql("SELECT * FROM v_fx_silver").write.format("delta").saveAsTable("silver.exchange_rates")
else:
    spark.sql("""
        MERGE INTO silver.exchange_rates AS target
        USING v_fx_silver AS source
        ON target.date = source.date
        WHEN MATCHED THEN UPDATE SET
            target.close_eurusd_x  = source.close_eurusd_x,
            target.close_iskusd_x  = source.close_iskusd_x,
            target.close__omx      = source.close__omx,
            target.high_eurusd_x   = source.high_eurusd_x,
            target.high_iskusd_x   = source.high_iskusd_x,
            target.high__omx       = source.high__omx,
            target.low_eurusd_x    = source.low_eurusd_x,
            target.low_iskusd_x    = source.low_iskusd_x,
            target.low__omx        = source.low__omx,
            target.open_eurusd_x   = source.open_eurusd_x,
            target.open_iskusd_x   = source.open_iskusd_x,
            target.open__omx       = source.open__omx,
            target.volume_eurusd_x = source.volume_eurusd_x,
            target.volume_iskusd_x = source.volume_iskusd_x,
            target.volume__omx     = source.volume__omx,
            target.ingested_at     = source.ingested_at,
            target.close_iskeur    = source.close_iskeur,
            target.processed_at    = source.processed_at
        WHEN NOT MATCHED THEN INSERT (
            date, close_eurusd_x, close_iskusd_x, close__omx,
            high_eurusd_x, high_iskusd_x, high__omx,
            low_eurusd_x, low_iskusd_x, low__omx,
            open_eurusd_x, open_iskusd_x, open__omx,
            volume_eurusd_x, volume_iskusd_x, volume__omx,
            ingested_at, close_iskeur, processed_at
        )
        VALUES (
            source.date, source.close_eurusd_x, source.close_iskusd_x, source.close__omx,
            source.high_eurusd_x, source.high_iskusd_x, source.high__omx,
            source.low_eurusd_x, source.low_iskusd_x, source.low__omx,
            source.open_eurusd_x, source.open_iskusd_x, source.open__omx,
            source.volume_eurusd_x, source.volume_iskusd_x, source.volume__omx,
            source.ingested_at, source.close_iskeur, source.processed_at
        )
    """)

print(f"Saved to silver.exchange_rates - {spark.table('silver.exchange_rates').count()} rows")